# 01 — Data Inventory & Feasibility

**Goal:** ground the SDVF (Solution / Desirability / Viability / Feasibility) assessment in evidence.
This notebook answers: *what do we actually have on disk*, *what's complete*, *what's joinable*,
and *which proposed features survive the data-feasibility filter*.

**Scope:** all 24 registered views in the DuckDB warehouse — GPS side (2025-03-18 sample), AFC side
(Jan–Mar 2026, ~17.8M boardings), and the reference layer (stops, routes, vehicles, drivers, **students,
schools, users** — newly added).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from src.duckdb_loader import BusDuckDB

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)
sns.set_theme(style='whitegrid', context='notebook')

db = BusDuckDB('../data', ':memory:')
db.register_views()
print('OK — DuckDB warehouse ready.')


INFO | DuckDB ready


INFO | View 'tickets' registered (90 file(s))


INFO | View 'payments' registered (90 file(s))


INFO | View 'gps_raw' registered


INFO | View 'gps_trips' registered


INFO | View 'gps_stop_times' registered


... (16 outputs left out)

INFO | View 'ticket_schools' registered


INFO | View 'stops' registered


INFO | View 'mapping' registered


OK — DuckDB warehouse ready.


## 1. Warehouse Overview

Every registered view, with its row count, ordered by size. This is the inventory.


In [2]:
views = db.show_views().to_pandas()
inv = []
for v in views['table_name']:
    try:
        n = db.row_count(v)
    except Exception as e:
        n = -1
    inv.append({'view': v, 'rows': n})
inv = pd.DataFrame(inv).sort_values('rows', ascending=False).reset_index(drop=True)
inv['rows_fmt'] = inv['rows'].map(lambda x: f'{x:,}' if x >= 0 else 'error')
inv


,view,rows,rows_fmt
0,tickets,17883623,"17,883,623"
1,payments,17882648,"17,882,648"
2,ticket_trips,1205482,"1,205,482"
3,gps_raw,720088,"720,088"
4,gps_extracted,369887,"369,887"
5,gps_schedules,283012,"283,012"
6,gps_stop_times,56969,"56,969"
7,ticket_route_stops,13226,"13,226"
8,ticket_stations,5502,"5,502"
9,stops,4710,"4,710"


In [3]:
# Visualise the inventory: rows per view on a log scale
fig = px.bar(
    inv.sort_values('rows'),
    x='rows', y='view', orientation='h', log_x=True,
    title='Warehouse inventory — rows per view (log scale)',
    labels={'rows':'Row count (log)', 'view':''},
    color='rows', color_continuous_scale='Teal',
    height=650,
)
fig.update_layout(coloraxis_showscale=False, margin=dict(l=140,r=40,t=60,b=40))
fig.show()


## 2. GPS coverage — temporal & route

GPS data is sampled in two layers:
- **raw_gps** — every 10-second ping for every vehicle on a single day (`2025-03-18`)
- **extracted** — HMM-filtered trips for a subset of routes


In [4]:
gps_dates = db.query('''
    SELECT MIN(date) AS start, MAX(date) AS end,
           COUNT(DISTINCT date) AS days,
           COUNT(DISTINCT vehicle) AS vehicles,
           COUNT(*) AS pings
    FROM gps_raw
''').to_pandas()
gps_dates


,start,end,days,vehicles,pings
0,2025-03-18,2025-03-18,1,131,720088


In [5]:
# Which routes appear in extracted (HMM-filtered) trips?
ext = db.query('''
    SELECT route_no, COUNT(DISTINCT trip_id) AS trips,
           COUNT(*) AS rows,
           MIN(date) AS first_date, MAX(date) AS last_date
    FROM gps_extracted
    GROUP BY route_no
    ORDER BY trips DESC
''').to_pandas()
print(f'Routes processed: {len(ext)}')
ext


Routes processed: 9


,route_no,trips,rows,first_date,last_date
0,151,201,47422,2025-03-18,2025-03-18
1,141,161,52936,2025-03-18,2025-03-18
2,99,141,47805,2025-03-18,2025-03-18
3,29,140,47042,2025-03-18,2025-03-18
4,41,126,43602,2025-03-18,2025-03-18
5,61,126,30494,2025-03-18,2025-03-18
6,84,120,27357,2025-03-18,2025-03-18
7,102,115,49216,2025-03-18,2025-03-18
8,57,114,24013,2025-03-18,2025-03-18


In [6]:
# Bus ping density by hour of day — does coverage span operating hours?
hourly = db.query('''
    SELECT EXTRACT(hour FROM datetime) AS hour, COUNT(*) AS pings
    FROM gps_raw
    GROUP BY hour ORDER BY hour
''').to_pandas()
fig = px.bar(
    hourly, x='hour', y='pings',
    title='GPS ping density by hour of day (2025-03-18) — operating window check',
    labels={'hour':'Hour (0–23)', 'pings':'GPS pings'},
    color='pings', color_continuous_scale='Sunset',
)
fig.update_layout(coloraxis_showscale=False, height=380)
fig.show()


## 3. AFC coverage — temporal, payment methods, ticket types

The AFC layer is the *biggest* dataset (~17.8M payments). This section maps its shape.


In [7]:
# AFC volume per day → did anything go missing?
daily = db.query('''
    SELECT DATE_TRUNC('day', TO_TIMESTAMP(time_created_ticket)) AS day,
           COUNT(*) AS tickets
    FROM tickets
    GROUP BY day ORDER BY day
''').to_pandas()
print(f'Date range: {daily["day"].min()} → {daily["day"].max()}  ({len(daily)} days)')
fig = px.line(
    daily, x='day', y='tickets', markers=True,
    title='Daily AFC ticket volume (Jan–Mar 2026)',
    labels={'day':'', 'tickets':'Tickets / day'},
)
fig.update_traces(line_color='#2c7fb8')
fig.update_layout(height=380)
fig.show()


Date range: 2026-01-01 00:00:00+07:00 → 2026-03-31 00:00:00+07:00  (90 days)


In [8]:
# Payment method breakdown
pay = db.query('''
    SELECT payment_method, COUNT(*) AS n, SUM(final_price)::BIGINT AS revenue
    FROM payments
    GROUP BY payment_method ORDER BY n DESC
''').to_pandas()
pay['share_%'] = (pay['n'] / pay['n'].sum() * 100).round(1)
pay


,payment_method,n,revenue,share_%
0,Tiền mặt,17196979,68994570000,96.2
1,Mastercard,263291,1305356710,1.5
2,Visa,196858,1079224690,1.1
3,CCCD > 60 tuổi,95786,425801000,0.5
4,Momo,85555,400900000,0.5
5,NAPAS,17265,98093200,0.1
6,ShopeePay,13101,74291000,0.1
7,ZaloPay,12198,62308000,0.1
8,VnptMoney,1517,9871000,0.0
9,ViettelMoney,97,621000,0.0


In [9]:
# Ticket type breakdown — uncovers student/free/promo segments
tt = db.query('''
    SELECT ticket_type, COUNT(*) AS n
    FROM tickets
    GROUP BY ticket_type ORDER BY n DESC LIMIT 15
''').to_pandas()
tt['share_%'] = (tt['n'] / tt['n'].sum() * 100).round(1)
fig = px.bar(
    tt.sort_values('n'), x='n', y='ticket_type', orientation='h',
    title='AFC ticket types — top 15',
    labels={'n':'Tickets', 'ticket_type':''},
    color='n', color_continuous_scale='Viridis', text='share_%',
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(coloraxis_showscale=False, height=520, margin=dict(l=200))
fig.show()


## 4. Reference layer — including the new students / schools tables

The reference tables are small but high-leverage. The **students → schools** chain unlocks
the flagship "school-aware demand" theme that no public bus analytics tool currently does.


In [10]:
ref = {
    'stops':            db.row_count('stops'),
    'gps_routes':       db.row_count('gps_routes'),
    'ticket_routes':    db.row_count('ticket_routes'),
    'gps_vehicles':     db.row_count('gps_vehicles'),
    'ticket_buses':     db.row_count('ticket_buses'),
    'gps_drivers':      db.row_count('gps_drivers'),
    'ticket_drivers':   db.row_count('ticket_drivers'),
    'ticket_users':     db.row_count('ticket_users'),
    'ticket_students':  db.row_count('ticket_students'),
    'ticket_schools':   db.row_count('ticket_schools'),
    'gps_companies':    db.row_count('gps_companies'),
    'ticket_companies': db.row_count('ticket_companies'),
}
pd.DataFrame.from_dict(ref, orient='index', columns=['rows']).sort_values('rows', ascending=False)


,rows
stops,4710
gps_vehicles,2550
ticket_buses,2490
gps_drivers,1764
ticket_students,1615
ticket_drivers,1173
ticket_users,952
gps_routes,184
ticket_routes,149
ticket_schools,60


In [11]:
# Top schools by student count — the demand-anchor of the flagship theme
top_schools = db.query('''
    SELECT sch.school_name,
           sch.school_address,
           COUNT(stu.student_id) AS students
    FROM ticket_students stu
    JOIN ticket_schools  sch USING (school_id)
    GROUP BY sch.school_name, sch.school_address
    ORDER BY students DESC
    LIMIT 12
''').to_pandas()
fig = px.bar(
    top_schools.sort_values('students'), x='students', y='school_name', orientation='h',
    title='Top schools by registered student-riders',
    labels={'students':'Students', 'school_name':''},
    color='students', color_continuous_scale='Plasma',
)
fig.update_layout(coloraxis_showscale=False, height=480, margin=dict(l=320))
fig.show()


## 5. Cross-side joinability

Critical SDVF check: can we link GPS trips to AFC trips? Per the dataset doc, **no** — the two
windows don't overlap. But the *route-level* and *station-level* dimensions DO align via the
shared route_no and station_id space.


In [12]:
# Date overlap test — should be ZERO days
overlap = db.query('''
    WITH gps_d AS (SELECT DISTINCT date AS d FROM gps_raw),
         afc_d AS (SELECT DISTINCT DATE_TRUNC('day', TO_TIMESTAMP(time_created_ticket))::DATE AS d FROM tickets)
    SELECT COUNT(*) AS overlapping_days
    FROM gps_d JOIN afc_d USING (d)
''').to_pandas()
print('Days where both GPS and AFC have data:', int(overlap['overlapping_days'][0]))

# Route overlap — share of route_no's present in both sides
route_overlap = db.query('''
    WITH g AS (SELECT DISTINCT route_no FROM gps_routes),
         t AS (SELECT DISTINCT route_no FROM ticket_routes)
    SELECT
      (SELECT COUNT(*) FROM g) AS gps_routes,
      (SELECT COUNT(*) FROM t) AS afc_routes,
      (SELECT COUNT(*) FROM g JOIN t USING (route_no)) AS shared_routes
''').to_pandas()
route_overlap


Days where both GPS and AFC have data: 0


,gps_routes,afc_routes,shared_routes
0,163,128,121


In [13]:
# Station overlap — by coordinates (round to 4 decimals = ~10 m)
station_overlap = db.query('''
    WITH g AS (
        SELECT DISTINCT ROUND(lat,4) AS la, ROUND(lng,4) AS lo FROM gps_stations WHERE lat IS NOT NULL
    ),
    t AS (
        SELECT DISTINCT ROUND(lat,4) AS la, ROUND(lng,4) AS lo FROM ticket_stations WHERE lat IS NOT NULL
    )
    SELECT
      (SELECT COUNT(*) FROM g) AS gps_stations,
      (SELECT COUNT(*) FROM t) AS afc_stations,
      (SELECT COUNT(*) FROM g JOIN t USING (la, lo)) AS shared_coords
''').to_pandas()
station_overlap


,gps_stations,afc_stations,shared_coords
0,4705,5449,4633


## 6. Completeness — null counts in key fields

The high-leverage joins all hinge on a few foreign keys (`trip_id`, `station_id`, `user_id`, `route_id`).
If those are sparse, downstream analytics break.


In [14]:
checks = [
    ('tickets',         ['trip_id', 'user_id', 'station_id', 'ticket_type', 'time_created_ticket']),
    ('payments',        ['ticket_id', 'final_price', 'payment_method']),
    ('ticket_trips',    ['trip_id', 'route_id', 'bus_id', 'direction']),
    ('gps_raw',         ['datetime', 'vehicle', 'lng', 'lat', 'speed']),
    ('gps_trips',       ['trip_id', 'route_id', 'driver_id', 'vehicle_id']),
    ('ticket_students', ['student_id', 'school_id']),
    ('ticket_users',    ['user_id', 'student_id']),
]
rows = []
for view, cols in checks:
    total = db.row_count(view)
    for c in cols:
        nulls = db.query(f'SELECT COUNT(*) AS n FROM {view} WHERE {c} IS NULL').to_pandas()['n'][0]
        rows.append({'view': view, 'field': c, 'total': total, 'nulls': int(nulls),
                     'null_pct': round(100 * nulls / max(total,1), 2)})
cmp_df = pd.DataFrame(rows)
cmp_df


,view,field,total,nulls,null_pct
0,tickets,trip_id,17883623,25149,0.14
1,tickets,user_id,17883623,17860514,99.87
2,tickets,station_id,17883623,1865,0.01
3,tickets,ticket_type,17883623,0,0.00
4,tickets,time_created_ticket,17883623,0,0.00
5,payments,ticket_id,17882648,0,0.00
6,payments,final_price,17882648,0,0.00
7,payments,payment_method,17882648,0,0.00
8,ticket_trips,trip_id,1205482,1527,0.13
9,ticket_trips,route_id,1205482,0,0.00


In [15]:
# Heatmap of null % across views × fields
pivot = cmp_df.pivot(index='view', columns='field', values='null_pct').fillna(-1)
fig = go.Figure(data=go.Heatmap(
    z=pivot.values, x=pivot.columns, y=pivot.index,
    colorscale=[[0,'#1b9e77'],[0.3,'#d9f0a3'],[0.7,'#fec44f'],[1,'#d7301f']],
    zmin=0, zmax=100, text=pivot.values.round(1), texttemplate='%{text}',
    colorbar=dict(title='null %'),
))
fig.update_layout(title='Null-rate heatmap — key fields per view', height=420)
fig.show()


## 7. Key findings — input to SDVF

1. **Two non-overlapping windows.** GPS (Mar 18, 2025 only — 50-day full set must be downloaded
   from Kaggle to unlock temporal features). AFC (Jan–Mar 2026, complete).
2. **Joinable at route × stop × hour-of-day**, NOT at trip_id. Design must enforce this invariant.
3. **Students → Schools chain is intact** and exposes a flagship angle: which schools generate
   which routes' demand, school-route dependency, summer-break sensitivity, equity audits.
4. **Payment methods are diverse** (cash, e-wallet, QR, EMV) — supports a "digital adoption" theme.
5. **GPS extracted set is narrow** — only a handful of routes processed; the rest of the 31 selected
   routes need to be re-processed once the full dataset is downloaded.
6. **Null rates in foreign keys** (`trip_id`, `user_id`, `station_id`) need a per-feature
   tolerance — features that depend on `user_id` will lose coverage where AFC records cash-only
   anonymous boardings.


## 8. Field-Level Description Tables (for report Ch 4.2)

For every column in the three major source families — **AFC tickets**, **GPS pings**,
and the **master tables (stops, routes)** — list the data type and a one-sentence
meaning. Tables are displayed here and exported to `notebooks/figures/fields_*.csv`
so the report can include them verbatim.


In [16]:
from pathlib import Path
import pandas as pd

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)


def schema_df(view_name: str) -> pd.DataFrame:
    """Return DESCRIBE result as a small pandas frame with field/type columns."""
    s = db.schema(view_name).to_pandas()
    return s[["column_name", "column_type"]].rename(
        columns={"column_name": "field", "column_type": "type"}
    )


# Manually authored one-line meanings, keyed by (view, field).
MEANINGS = {
    # --- AFC tickets ---
    ("tickets", "ticket_id"):           "Surrogate primary key for the boarding event.",
    ("tickets", "ticket_uuid"):         "Globally unique UUID for the ticket.",
    ("tickets", "ticket_code"):         "Human-readable ticket code printed on the receipt.",
    ("tickets", "ticket_series"):       "Paper-ticket batch / series identifier (mostly null for e-tickets).",
    ("tickets", "ticket_type"):         "Vietnamese fare-product label (e.g. 'Vé TM lượt HS-SV' = student cash single).",
    ("tickets", "trip_id"):             "FK to ticket_trips — the bus trip on which the boarding occurred.",
    ("tickets", "user_id"):             "Registered passenger ID; populated for ~0.1% of tickets only.",
    ("tickets", "user_id_1"):           "Secondary user ID column (legacy duplicate of user_id).",
    ("tickets", "station_id"):          "FK to ticket_stations — the boarding stop.",
    ("tickets", "lat"):                 "Latitude of the boarding event (decimal degrees, WGS-84).",
    ("tickets", "lng"):                 "Longitude of the boarding event (decimal degrees, WGS-84).",
    ("tickets", "time_created_ticket"): "Unix epoch seconds when the ticket was issued (boarding timestamp).",
    ("tickets", "aggregation_status"):  "Backend rollup status flag ('Thành công' = successfully aggregated).",
    ("tickets", "updated_at"):          "Unix epoch seconds of the last record update.",
    ("tickets", "month"):               "Partition tag (e.g. 'jan_2026') derived from source folder name.",

    # --- GPS pings ---
    ("gps_raw", "datetime"):  "Timestamp of the GPS ping (server-side, second precision).",
    ("gps_raw", "date"):      "Calendar date of the ping (denormalised partition key).",
    ("gps_raw", "vehicle"):   "Vehicle identifier (license plate or anonymised hash).",
    ("gps_raw", "lng"):       "Longitude of the bus at ping time (WGS-84).",
    ("gps_raw", "lat"):       "Latitude of the bus at ping time (WGS-84).",
    ("gps_raw", "driver"):    "Driver license / staff identifier on duty at ping time.",
    ("gps_raw", "speed"):     "Instantaneous speed reported by the on-board unit (km/h).",
    ("gps_raw", "door_up"):   "Boolean — front door open at ping time (used as boarding proxy).",
    ("gps_raw", "door_down"): "Boolean — rear door open at ping time (used as alighting proxy).",

    # --- Master tables ---
    ("stops", "stopId"):     "Surrogate primary key for the stop (city-wide master list).",
    ("stops", "code"):       "Public-facing stop code displayed at the shelter.",
    ("stops", "stop_name"):  "Human-readable stop name (Vietnamese, including diacritics).",
    ("stops", "type"):       "Stop type tag (e.g. 'BusStop', 'Terminal', 'MetroFeeder').",
    ("stops", "zone"):               "Administrative zone / district code.",
    ("stops", "ward"):               "Ward (phường) name.",
    ("stops", "address"):            "Free-text address string.",
    ("stops", "street"):             "Street name on which the stop sits.",
    ("stops", "support_disability"): "Accessibility flag — 1 if the stop is wheelchair-friendly.",
    ("stops", "status"):             "Operational status flag ('active', 'inactive', etc.).",
    ("stops", "lng"):        "Longitude of the stop (WGS-84).",
    ("stops", "lat"):        "Latitude of the stop (WGS-84).",
    ("stops", "search"):     "Pre-computed lowercase search string for fuzzy matching.",
    ("stops", "routes"):     "Comma-separated list of route_no values that serve the stop.",

    ("ticket_routes", "route_id"):   "Surrogate primary key for the AFC route record.",
    ("ticket_routes", "route_no"):   "Public route number (e.g. '152') — the canonical join key with GPS.",
    ("ticket_routes", "route_name"): "Origin → destination label shown to passengers.",
    ("ticket_routes", "company_id"): "FK to ticket_companies — the operator running the route.",
}


def annotate(view: str) -> pd.DataFrame:
    df = schema_df(view).copy()
    df["meaning"] = df["field"].map(lambda f: MEANINGS.get((view, f), ""))
    return df


fields_afc     = annotate("tickets")
fields_gps     = annotate("gps_raw")
fields_stops   = annotate("stops")
fields_routes  = annotate("ticket_routes")
fields_masters = pd.concat(
    [fields_stops.assign(table="stops"),
     fields_routes.assign(table="ticket_routes")],
    ignore_index=True,
)[["table", "field", "type", "meaning"]]

fields_afc.to_csv(FIG_DIR / "fields_afc.csv", index=False)
fields_gps.to_csv(FIG_DIR / "fields_gps.csv", index=False)
fields_masters.to_csv(FIG_DIR / "fields_masters.csv", index=False)

print(f"Wrote {FIG_DIR/'fields_afc.csv'}      ({len(fields_afc)} rows)")
print(f"Wrote {FIG_DIR/'fields_gps.csv'}      ({len(fields_gps)} rows)")
print(f"Wrote {FIG_DIR/'fields_masters.csv'}  ({len(fields_masters)} rows)")


Wrote figures/fields_afc.csv      (15 rows)
Wrote figures/fields_gps.csv      (9 rows)
Wrote figures/fields_masters.csv  (18 rows)


In [17]:
print("AFC tickets — field description")
fields_afc


AFC tickets — field description


,field,type,meaning
0,ticket_id,BIGINT,Surrogate primary key for the boarding event.
1,ticket_uuid,VARCHAR,Globally unique UUID for the ticket.
2,ticket_code,VARCHAR,Human-readable ticket code printed on the rece...
3,ticket_series,VARCHAR,Paper-ticket batch / series identifier (mostly...
4,ticket_type,VARCHAR,Vietnamese fare-product label (e.g. 'Vé TM lượ...
5,trip_id,DOUBLE,FK to ticket_trips — the bus trip on which the...
6,user_id,VARCHAR,Registered passenger ID; populated for ~0.1% o...
7,user_id_1,VARCHAR,Secondary user ID column (legacy duplicate of ...
8,station_id,DOUBLE,FK to ticket_stations — the boarding stop.
9,lat,DOUBLE,Latitude of the boarding event (decimal degree...


In [18]:
print("GPS pings — field description")
fields_gps


GPS pings — field description


,field,type,meaning
0,datetime,TIMESTAMP,"Timestamp of the GPS ping (server-side, second..."
1,date,DATE,Calendar date of the ping (denormalised partit...
2,vehicle,VARCHAR,Vehicle identifier (license plate or anonymise...
3,lng,DOUBLE,Longitude of the bus at ping time (WGS-84).
4,lat,DOUBLE,Latitude of the bus at ping time (WGS-84).
5,driver,VARCHAR,Driver license / staff identifier on duty at p...
6,speed,DOUBLE,Instantaneous speed reported by the on-board u...
7,door_up,BOOLEAN,Boolean — front door open at ping time (used a...
8,door_down,BOOLEAN,Boolean — rear door open at ping time (used as...


In [19]:
print("Master tables — field description")
fields_masters


Master tables — field description


,table,field,type,meaning
0,stops,stopId,BIGINT,Surrogate primary key for the stop (city-wide ...
1,stops,code,VARCHAR,Public-facing stop code displayed at the shelter.
2,stops,stop_name,VARCHAR,"Human-readable stop name (Vietnamese, includin..."
3,stops,type,VARCHAR,"Stop type tag (e.g. 'BusStop', 'Terminal', 'Me..."
4,stops,zone,VARCHAR,Administrative zone / district code.
5,stops,ward,VARCHAR,Ward (phường) name.
6,stops,address,VARCHAR,Free-text address string.
7,stops,street,VARCHAR,Street name on which the stop sits.
8,stops,support_disability,VARCHAR,Accessibility flag — 1 if the stop is wheelcha...
9,stops,status,VARCHAR,"Operational status flag ('active', 'inactive',..."
